In [ ]:
import praw
import pandas as pd
import os
from dotenv import load_dotenv
import logging
import time
from prawcore import exceptions as praw_exceptions

/Users/hachikaruanyakwee/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
#Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [3]:
#Load API credentials from .env file
load_dotenv()
REDDIT_CLIENT_ID = os.getenv("REDDIT_CLIENT_ID")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
REDDIT_USER_AGENT = os.getenv("REDDIT_USER_AGENT")

In [ ]:
#Authenticate with Reddit API
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT
)
logging.info("Successfully authenticated with Reddit API.")

2025-03-17 11:47:14,863 - INFO - Successfully authenticated with Reddit API.


In [5]:
#Define search parameters
subreddit_name = "canada"
search_query = "Mark Carney"
max_comments = 500
comments_data = []
submissions_processed = 0
submission_limit = 50  #Limit the number of submissions

In [6]:
#Fetch posts and comments with error handling and rate limit management
for submission in reddit.subreddit(subreddit_name).search(search_query, limit=submission_limit):
    logging.info(f"Processing submission: {submission.title} (ID: {submission.id})")
    try:
        submission.comments.replace_more(limit=0)  # Load all comments (can be resource-intensive)
        for comment in submission.comments.list():
            if len(comments_data) >= max_comments:
                break
            comments_data.append([comment.created_utc, comment.body, "Reddit"])
        submissions_processed += 1
        time.sleep(1)  # Be respectful to the API
    except praw_exceptions.TooManyRequests as e:
        logging.warning(f"Rate limit exceeded: {e}. Waiting for 60 seconds...")
        time.sleep(60)
        continue
    except Exception as e:
        logging.error(f"An error occurred while processing submission {submission.id}: {e}")

    if len(comments_data) >= max_comments:
        logging.info(f"Reached the maximum number of comments: {max_comments}")
        break

2025-03-17 11:48:48,794 - INFO - Processing submission: Mark Carney is the new Liberal leader, replacing Justin Trudeau - National | Globalnews.ca (ID: 1j7jo36)
2025-03-17 11:48:52,516 - INFO - Processing submission: Mark Carney elected Liberal leader (ID: 1j7jpfw)
2025-03-17 11:48:54,491 - INFO - Reached the maximum number of comments: 500


In [5]:
#Fetch posts
for submission in reddit.subreddit(subreddit_name).search(search_query, limit=50):
    submission.comments.replace_more(limit=0)  #Load all comments
    for comment in submission.comments.list():
        if len(data) >= max_comments:
            break
        data.append([comment.created_utc, comment.body, "Reddit"])


In [7]:
#Save to CSV
df = pd.DataFrame(comments_data, columns=["timestamp", "comment", "source"])

In [8]:
#Define the correct relative path to the existing 'data' folder at the repo root
data_dir = os.path.join("..", "data")  # Go up one level, then into the 'data' folder

In [9]:
#Ensure the 'data' directory exists before saving the file
os.makedirs(data_dir, exist_ok=True)

In [11]:
#Save to CSV
output_filepath = os.path.join(data_dir, "reddit_data_comments.csv")
df.to_csv(output_filepath, index=False)

logging.info(f"Successfully collected {len(df)} comments and saved to {output_filepath}!")
print(f"✅ Successfully collected {len(df)} Reddit comments and saved to {output_filepath}!")

2025-03-17 11:50:29,442 - INFO - Successfully collected 500 comments and saved to ../data/reddit_data_comments.csv!


✅ Successfully collected 500 Reddit comments and saved to ../data/reddit_data_comments.csv!
